## Evaluación de modelos

### 1. Importaciones

In [1]:
from fastai.tabular.all import *
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
from sklearn.linear_model import LinearRegression
from sklearn.neighbors import KNeighborsRegressor
from sklearn.ensemble import (
    AdaBoostRegressor, GradientBoostingRegressor,
    BaggingRegressor, RandomForestRegressor
)
from sklearn.tree import DecisionTreeRegressor
from lightgbm import LGBMRegressor
import xgboost as xgb
from catboost import CatBoostRegressor
import re

pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)
pd.set_option('display.expand_frame_repr', False)

## 2. Evaluador de modelos

In [2]:
from sklearn.metrics import mean_absolute_percentage_error, mean_squared_log_error, mean_squared_error
import numpy as np

# Funciones auxiliares
def mape_percent(y_true, y_pred):
    # MAPE en porcentaje
    return 100 * mean_absolute_percentage_error(y_true, y_pred)

def rmsle(y_true, y_pred):
    # MSLE requiere no-negativos
    y_true_safe = np.maximum(y_true, 0)
    y_pred_safe = np.maximum(y_pred, 0)
    return np.sqrt(mean_squared_log_error(y_true_safe, y_pred_safe))

def root_mean_squared_error(y_true, y_pred):
    return np.sqrt(mean_squared_error(y_true, y_pred))

# Evaluación de modelos
def fit_transform_model(X, y):
    # Split train/test
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42
    )

    # Lista de modelos
    models = [
        ("LinearRegression", LinearRegression(n_jobs=-1)),
        ("KNeighborsRegressor", KNeighborsRegressor(n_neighbors=5, n_jobs=-1)),
        ("AdaBoostRegressor", AdaBoostRegressor(n_estimators=100, learning_rate=0.1, random_state=42)),
        ("DecisionTreeRegressor", DecisionTreeRegressor(max_depth=10, random_state=42)),
        ("GradientBoostingRegressor", GradientBoostingRegressor(n_estimators=100, learning_rate=0.1, max_depth=3, random_state=42)),
        ("BaggingRegressor", BaggingRegressor(n_estimators=50, n_jobs=-1, random_state=42)),
        ("RandomForestRegressor", RandomForestRegressor(n_estimators=100, max_depth=10, n_jobs=-1, random_state=42)),
        ("LGBMRegressor", LGBMRegressor(n_estimators=200, learning_rate=0.1, max_depth=-1, n_jobs=-1, random_state=42, verbose=-1)),
        ("XGBRegressor", xgb.XGBRegressor(n_estimators=200, learning_rate=0.1, max_depth=6,
                         subsample=0.8, colsample_bytree=0.8, n_jobs=-1,
                         tree_method="hist", random_state=42, verbosity=0)),
        ("CatBoostRegressor", CatBoostRegressor(iterations=200, depth=6, learning_rate=0.1, verbose=False, random_state=42))
    ]
    
    return [(name, model.fit(X_train, y_train).predict(X_test), y_test) 
            for name, model in models]

### 3. Import dataset

In [3]:
# Cargar el objeto TabularPandas previamente guardado
to = load_pickle('./df_train-tabular-object.pkl')

## 4. Fit

In [4]:
# Cargar el objeto TabularPandas desde el pickle
to = load_pickle('./df_train-tabular-object.pkl')

# Extraer features y target del conjunto de entrenamiento
X_train = to.train.xs.copy()
y_train = to.train.y.copy()

# Muestreo (opcional, si necesitas trabajar con una muestra más pequeña)
np.random.seed(42)  # Para reproducibilidad
sample_frac = 0.20
sampled_indices = np.random.choice(len(X_train), size=int(len(X_train) * sample_frac), replace=False)
X_train_sample = X_train.iloc[sampled_indices]
y_train_sample = y_train.iloc[sampled_indices]

# Entrenar modelos y obtener predicciones
results = fit_transform_model(X_train_sample, y_train_sample)

# Imprimir resultados una sola vez
print(f"\n{'Modelo':<25} | {'RMSLE':>10} | {'RMSE':>10}")
print("-" * 50)
for name, y_pred, y_true in results:
    rmsle_val = rmsle(y_true, y_pred)
    rmse_val = root_mean_squared_error(y_true, y_pred)
    print(f"{name:<25} | {rmsle_val:>10.4f} | {rmse_val:>10.4f}")


Modelo                    |      RMSLE |       RMSE
--------------------------------------------------
LinearRegression          |     1.0334 |  1013.1543
KNeighborsRegressor       |     0.9951 |  1020.2096
AdaBoostRegressor         |     1.0503 |   890.3532
DecisionTreeRegressor     |     0.5216 |   699.5919
GradientBoostingRegressor |     0.6527 |   657.6321
BaggingRegressor          |     0.4607 |   570.6674
RandomForestRegressor     |     0.5067 |   654.3548
LGBMRegressor             |     0.5860 |   570.6508
XGBRegressor              |     0.6473 |   570.9918
CatBoostRegressor         |     0.5900 |   605.8965
